# 07b — Challenger Models

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
from scorelab import tuning as T, explain as E, survival as SV, causal as CA
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

Whether any model family beats the card out of time by enough to justify its explainability cost, once each one is tuned honestly. The challengers are fitted on the training vintages: XGBoost (tuned with Optuna and at the NB07 defaults), LightGBM, random forest, two EBMs (monotone with main effects only, and unconstrained with pairwise terms), two small MLPs (each tuned with Optuna) and a stacked ensemble. **All tuning, early stopping and the stack's meta-model use the 2015 validation vintage only; 2016 is touched once, in the comparison table.**

**Inputs are held fixed so model and input-set effects can be told apart.** Every tree model, the EBMs and the first MLP see the same NB03 candidate attributes (the trees handle missing values natively; the MLP gets them standardised, median-imputed, with a missing-indicator column per attribute that has missing values, all fitted on 2012–2014 only). The second MLP sees the NB04 WOE attributes, the card's own coding. The `inputs` column of the comparison table states each model's input set.

Decisions: **W — the model set and the search budget**, **X — which challenger goes forward** (to NB07c for explanation, and to the champion memo in NB07).

In [ ]:
frame = load_frame()
tr, va, oot = (frame[frame.split == s] for s in ("train", "validate", "oot"))
card, specs = S.load_card(MODELS / "champion"); card_attrs = card.attributes
cands = json.loads((MODELS / "candidate_attributes.json").read_text(encoding="utf-8"))["candidates"]
print(f"train {len(tr):,} · validate {len(va):,} · oot {len(oot):,} · {len(cands)} candidates · card on {len(card_attrs)}")

p_card = {"va": S.score(va, card, specs)["p_bad"].to_numpy(), "oot": S.score(oot, card, specs)["p_bad"].to_numpy()}
bins = B.load_specs(MODELS / "bins.json")              # NB04 bins: the WOE inputs option for the MLP

### Decision W — Model set and search budget

Which families earn a place in the comparison, and how much search each gets. A GBM trial on the full training set takes 20–60 s, an MLP trial a few minutes. Is the search budget enough (the spread of the top-5 validation AUCs below tells you whether the search has flattened)? Which models must keep monotone constraints for reason codes to be defensible? Read the two MLPs together: the raw-input MLP against the GBM isolates the model family on the same inputs; the WOE MLP against the raw MLP isolates the input set for the same family.

**Your reasoning:**

_(write here)_

In [ ]:
N_TRIALS_XGB, N_TRIALS_MLP = 30, 20      # Optuna budgets per search (validation AUC on 2015); each MLP gets its own
woe_attrs = [a for a in cands if a in bins]
MLP_RAW, MLP_WOE = f"MLP (raw, {len(cands)} candidates)", f"MLP (WOE, {len(woe_attrs)} binned)"
RAW = f"raw, {len(cands)} NB03 candidates"
INPUTS = {
    "Scorecard (champion)": f"WOE, {len(card_attrs)} card attributes",
    "XGBoost (tuned, monotone)": RAW + " (missing native)", "XGBoost (NB07 defaults)": RAW + " (missing native)",
    "LightGBM (monotone)": RAW + " (missing native)", "Random forest (monotone)": RAW + " (missing native)",
    "EBM (monotone, main effects)": RAW + " (missing native)", "EBM (10 pairs, unconstrained)": RAW + " (missing native)",
    MLP_RAW: RAW + ", standardised, median-imputed + missing flags (fit on 2012-14)",
    MLP_WOE: f"WOE, {len(woe_attrs)} NB04 binned attributes",
}
fits, times = {}, {}

In [ ]:
# Gradient boosting: Optuna search on 2015, then the tuned model, the NB07 default and LightGBM
best_xgb, trials_xgb = T.tune_xgb(tr, va, cands, n_trials=N_TRIALS_XGB)
display(trials_xgb.head(10))
print("spread of the top-5 validation AUCs:", round(T.best_trial_gap(trials_xgb), 4))
xgb_params = {k: v for k, v in best_xgb.items() if k != "n_rounds"}
fits["XGBoost (tuned, monotone)"], times["XGBoost (tuned, monotone)"] = C.timed(
    C.fit_xgb, tr, tr["bad"], va, va["bad"], cands, monotone=True, params=xgb_params)
fits["XGBoost (NB07 defaults)"], times["XGBoost (NB07 defaults)"] = C.timed(
    C.fit_xgb, tr, tr["bad"], va, va["bad"], cands, monotone=True)
fits["LightGBM (monotone)"], times["LightGBM (monotone)"] = C.timed(
    C.fit_lgbm, tr, tr["bad"], va, va["bad"], cands, monotone=True)

In [ ]:
# Random forest and the two EBMs
fits["Random forest (monotone)"], times["Random forest (monotone)"] = C.timed(
    C.fit_random_forest, tr, tr["bad"], cands, monotone=True)
fits["EBM (monotone, main effects)"], times["EBM (monotone, main effects)"] = C.timed(
    C.fit_ebm, tr, tr["bad"], cands, monotone=True)
fits["EBM (10 pairs, unconstrained)"], times["EBM (10 pairs, unconstrained)"] = C.timed(
    C.fit_ebm, tr, tr["bad"], cands, monotone=False, interactions=10)

In [ ]:
# Two MLPs, same Optuna budget and early stopping on 2015: raw candidates vs WOE inputs
best_mlp = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for (name, attrs_m, specs_m), ax in zip(((MLP_RAW, cands, None), (MLP_WOE, woe_attrs, bins)), axes):
    best_mlp[name], trials_m = T.tune_mlp(tr, va, attrs_m, specs=specs_m, n_trials=N_TRIALS_MLP)
    print(name); display(trials_m.head(5))
    fits[name], times[name] = C.timed(C.fit_mlp, tr, tr["bad"], va, va["bad"], attrs_m, specs=specs_m, **best_mlp[name])
    h = fits[name].history
    ax.plot(h.epoch, h.train_loss, label="train loss"); ax.plot(h.epoch, h.valid_log_loss, label="2015 log-loss")
    ax.axvline(fits[name].best_epoch, ls="--", c="grey"); ax.set_xlabel("epoch"); ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()
print("missing-indicator columns in the raw MLP:", fits[MLP_RAW].scaler.flagged)

In [ ]:
# Stacked ensemble: meta-model on 2015 predictions of models trained on 2012-2014
STACK_BASE = ["XGBoost (tuned, monotone)", "Random forest (monotone)", "EBM (monotone, main effects)", MLP_RAW]
INPUTS["Stacked ensemble"] = "log-odds of: " + ", ".join(STACK_BASE)
base_va = {n: C.predict_p(fits[n], va) for n in STACK_BASE}
stack, times["Stacked ensemble"] = C.timed(C.fit_stack, base_va, va["bad"])
display(pd.Series(stack.weights, name="weight on log-odds").round(3))

In [ ]:
# The comparison on 2016: every model against the champion card, one table
preds = {"Scorecard (champion)": p_card["oot"]}
preds.update({n: C.predict_p(m, oot) for n, m in fits.items()})
preds["Stacked ensemble"] = stack.predict({n: preds[n] for n in STACK_BASE})
NOTES = {
    "Scorecard (champion)": "points table; exact reason codes (points lost vs best bin); monotone by construction",
    "XGBoost (tuned, monotone)": "monotone; exact TreeSHAP reasons; interactions need SHAP to read",
    "XGBoost (NB07 defaults)": "as tuned XGBoost",
    "LightGBM (monotone)": "monotone; exact TreeSHAP reasons; interactions need SHAP to read",
    "Random forest (monotone)": "hundreds of deep trees; SHAP only; probabilities usually need recalibration",
    "EBM (monotone, main effects)": "one shape function per attribute; reads like a card; exact additive reasons",
    "EBM (10 pairs, unconstrained)": "shape functions plus 10 pairwise tables; readable, not monotone",
    MLP_RAW: "black box; post-hoc explanations only; no monotonicity guarantee",
    MLP_WOE: "black box on the card's bins; post-hoc explanations only; no monotonicity guarantee",
    "Stacked ensemble": "weighted log-odds of four models; as explainable as its least explainable part",
}
fit_s = {**times, "Scorecard (champion)": np.nan}
cmp = V.model_comparison(oot["bad"], preds, reference="Scorecard (champion)", fit_seconds=fit_s, notes=NOTES, inputs=INPUTS)
cmp = cmp.sort_values("gini", ascending=False).reset_index(drop=True)
(config.DOCS_DIR / "exhibits").mkdir(parents=True, exist_ok=True)
cmp.to_csv(config.DOCS_DIR / "exhibits" / "challenger_comparison_oot.csv", index=False)
display(cmp)

In [ ]:
# Gini with DeLong 95% intervals on 2016
c = cmp.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(c) + 1))
ax.errorbar(c.gini, range(len(c)), xerr=[c.gini - c.gini_ci_low, c.gini_ci_high - c.gini], fmt="o", capsize=3)
ax.axvline(cmp.set_index("model").loc["Scorecard (champion)", "gini"], ls="--", c="grey")
ax.set_yticks(range(len(c))); ax.set_yticklabels(c.model); ax.set_xlabel("Gini on 2016 (DeLong 95% CI)"); plt.tight_layout(); plt.show()

### Decision X — Which challenger goes forward

Read the table as a lender would: is the best challenger's Gini lift over the card outside the card's interval (`delong_p_vs_ref`)? Does the lift survive calibration (slope near 1, small calibration-in-the-large) or does the model need a recalibration layer? Is the EBM close enough to the GBM that an additive, readable model is the better trade? Does the stack add anything over its best component? Which model goes to NB07c for explanation (the tuned XGBoost by default, because TreeSHAP is exact for it)?

**Your reasoning:**

_(write here)_

In [ ]:
BEST = "XGBoost (tuned, monotone)"        # the challenger NB07c explains (must be an XGBoost model for exact SHAP)
C.save_xgb(fits[BEST], MODELS / "challenger_xgb_tuned.json")
(MODELS / "challenger_tuning.json").write_text(json.dumps({"best": BEST, "xgb_params": xgb_params, "xgb_n_rounds": best_xgb["n_rounds"],
           "mlp_params": {n: {k: list(v) if isinstance(v, tuple) else v for k, v in p.items()} for n, p in best_mlp.items()},
           "inputs": INPUTS,
           "stack_weights": stack.weights, "n_trials": {"xgb": N_TRIALS_XGB, "mlp": N_TRIALS_MLP},
           "ranking_oot": cmp[["model", "gini", "gini_ci_low", "gini_ci_high"]].to_dict(orient="records")}, indent=1, default=float), encoding="utf-8")
print("saved", MODELS / "challenger_xgb_tuned.json")